# Hey Vega: synthetic-data training preparation
Prepared 28 October 2026. Experimental, NOT executed or validated on Colab yet.

Uses upstream openWakeWord under Apache-2.0; original notebook attribution retained. Source: https://github.com/dscripka/openWakeWord/tree/368c03716d1e92591906a84949bc477f3a834455 . Changes: isolated Python3.10 setup, Hey Vega configuration, pinned repositories and ONNX-only final artifact target.

Choose a free GPU runtime if available. This notebook downloads public datasets (one feature file is17.28GB, plus speech/noise assets) and trains an experimental candidate. It does not mount Drive, request secrets or upload owner recordings. Check dataset/voice licenses before redistributing artifacts. Keep your local validation-01 recordings separate; they are not inputs to training.

This environment uses legacy upstream dependencies. Compatibility is unverified; installation/import checks stop on errors. A trained artifact is still subject to local false-wake, recall and live desktop tests. No ChatGPT-quality or training-time guarantee is made.


In [ ]:
import sys, subprocess, pathlib, os, shutil, platform, json
assert platform.system() == "Linux", "Use a fresh Linux Colab runtime"
assert pathlib.Path("/content").is_dir(), "This wrapper is intended for Colab"
assert shutil.disk_usage("/content").free >= 60 * 1024**3, "Need at least60GiB free for datasets and intermediate files"
print("Large public-data downloads; owner recordings stay local.")
assert input("Type PREPARE to install the isolated training environment: ").strip() == "PREPARE"
subprocess.run([sys.executable,"-m","pip","install","uv==0.9.5"],check=True)
subprocess.run([sys.executable,"-m","uv","venv","--python","3.10","--seed","/content/vega-env"],check=True)
PY="/content/vega-env/bin/python"
WORK=pathlib.Path("/content/vega-training"); WORK.mkdir(exist_ok=True)
def pip(*packages):
    subprocess.run([sys.executable,"-m","uv","pip","install","--python",PY,*packages],check=True)
pip("torch==2.5.1","torchaudio==2.5.1","--index-url","https://download.pytorch.org/whl/cu121")
pip("numpy==1.23.5","scipy==1.10.1","pyarrow==14.0.2","protobuf==3.19.6","tensorflow-cpu==2.8.1","tensorflow-probability==0.16.0","onnx==1.12.0","onnx-tf==1.10.0","onnxruntime==1.17.3","tflite-runtime==2.14.0","scikit-learn==1.3.2","requests==2.32.3","ipykernel==6.29.5","nbconvert==7.16.6","piper-phonemize==1.1.0","webrtcvad==2.0.10","mutagen==1.47.0","torchinfo==1.8.0","torchmetrics==1.2.0","speechbrain==0.5.14","audiomentations==0.33.0","torch-audiomentations==0.11.0","acoustics==0.2.6","pronouncing==0.2.0","datasets==2.14.6","deep-phonemizer==0.0.19","pyyaml==6.0.2")
for name,url,commit in [("openwakeword","https://github.com/dscripka/openWakeWord","368c03716d1e92591906a84949bc477f3a834455"),("piper-sample-generator","https://github.com/rhasspy/piper-sample-generator","2971426a55072f7d22fec416ca7800df8bd23207")]:
    destination=WORK/name
    if not destination.exists():subprocess.run(["git","clone",url,str(destination)],check=True)
    subprocess.run(["git","-C",str(destination),"checkout","--detach",commit],check=True)
pip("--no-deps","-e",str(WORK/"openwakeword"))
subprocess.run([PY,"-m","pip","check"],check=True)
subprocess.run([PY,"-c","import torch,torchaudio,tensorflow,datasets,openwakeword; print('Training imports passed; GPU:',torch.cuda.is_available())"],check=True)
subprocess.run([PY,"-m","ipykernel","install","--user","--name","vega-training-310"],check=True)
with (WORK/"environment-freeze.txt").open("w") as out:subprocess.run([PY,"-m","pip","freeze"],stdout=out,check=True)
print("Environment prepared. No owner audio uploaded. Continue only if all checks passed.")


## Public assets and experimental training
The next cell can run for hours and may exceed a free session. It contains the pinned upstream data preparation and training workflow. Do not enable paid compute automatically.


In [ ]:
PREPARED_NOTEBOOK_JSON = '{"cells": [{"cell_type": "markdown", "id": "c1eab0b3", "metadata": {"id": "c1eab0b3"}, "source": ["# Introduction"]}, {"cell_type": "markdown", "id": "882058c5", "metadata": {"id": "882058c5"}, "source": ["This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.\\n", "\\n", "Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.\\n", "\\n", "At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:\\n", "\\n", "- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data\\n", "- Variable learning rates with cosine decay and multiple cycles\\n", "- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training\\n", "- Cycical weight schedules for negative examples to help the model reduce false-positive rates\\n", "\\n", "See the contents of the `train.py` file for more details."]}, {"cell_type": "markdown", "id": "e08d031b", "metadata": {"id": "e08d031b"}, "source": ["# Environment Setup"]}, {"cell_type": "markdown", "id": "aee78c37", "metadata": {"id": "aee78c37"}, "source": ["To begin, we\'ll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.\\n", "\\n", "**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested."]}, {"cell_type": "code", "execution_count": null, "id": "4b1227eb", "metadata": {"id": "4b1227eb"}, "outputs": [], "source": ["# Dependencies and pinned repositories are provisioned by the VEGA wrapper.\\n"]}, {"cell_type": "code", "execution_count": null, "id": "d4c1056e", "metadata": {"ExecuteTime": {"end_time": "2023-09-04T13:42:01.183840Z", "start_time": "2023-09-04T13:41:59.752153Z"}, "id": "d4c1056e"}, "outputs": [], "source": ["# Imports\\n", "\\n", "import os\\n", "import numpy as np\\n", "import torch\\n", "import sys\\n", "from pathlib import Path\\n", "import uuid\\n", "import yaml\\n", "import datasets\\n", "import scipy\\n", "from tqdm import tqdm\\n"]}, {"cell_type": "markdown", "id": "e9d7a05a", "metadata": {"id": "e9d7a05a"}, "source": ["# Download Data"]}, {"cell_type": "markdown", "id": "c52f75cc", "metadata": {"id": "c52f75cc"}, "source": ["When training new openWakeWord models using the automated procedure, four specific types of data are required:\\n", "\\n", "1) Synthetic examples of the target word/phrase generated with text-to-speech models\\n", "\\n", "2) Synthetic examples of adversarial words/phrases generated with text-to-speech models\\n", "\\n", "3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic\\n", "\\n", "4) Generic \\"negative\\" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.\\n", "\\n", "5) Validation data to use for early-stopping when training the model.\\n", "\\n", "For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs."]}, {"cell_type": "code", "execution_count": null, "id": "d25a93b1", "metadata": {"ExecuteTime": {"end_time": "2023-09-04T01:07:17.746749Z", "start_time": "2023-09-04T01:07:17.740846Z"}, "id": "d25a93b1"}, "outputs": [], "source": ["# Download room impulse responses collected by MIT\\n", "# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html\\n", "\\n", "output_dir = \\"./mit_rirs\\"\\n", "if not os.path.exists(output_dir):\\n", "    os.mkdir(output_dir)\\n", "rir_dataset = datasets.load_dataset(\\"davidscripka/MIT_environmental_impulse_responses\\", split=\\"train\\", streaming=True)\\n", "\\n", "# Save clips to 16-bit PCM wav files\\n", "for row in tqdm(rir_dataset):\\n", "    name = row[\'audio\'][\'path\'].split(\'/\')[-1]\\n", "    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row[\'audio\'][\'array\']*32767).astype(np.int16))"]}, {"cell_type": "code", "execution_count": null, "id": "2c0e178b", "metadata": {"id": "2c0e178b"}, "outputs": [], "source": ["## Download noise and background audio\\n", "\\n", "# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)\\n", "# Download one part of the audioset .tar files, extract, and convert to 16khz\\n", "# For full-scale training, it\'s recommended to download the entire dataset from\\n", "# https://huggingface.co/datasets/agkphysics/AudioSet, and\\n", "# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)\\n", "\\n", "if not os.path.exists(\\"audioset\\"):\\n", "    os.mkdir(\\"audioset\\")\\n", "\\n", "fname = \\"bal_train09.tar\\"\\n", "out_dir = f\\"audioset/{fname}\\"\\n", "link = \\"https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/\\" + fname\\n", "!wget -O {out_dir} {link}\\n", "!cd audioset && tar -xvf bal_train09.tar\\n", "\\n", "output_dir = \\"./audioset_16k\\"\\n", "if not os.path.exists(output_dir):\\n", "    os.mkdir(output_dir)\\n", "\\n", "# Convert audioset files to 16khz sample rate\\n", "audioset_dataset = datasets.Dataset.from_dict({\\"audio\\": [str(i) for i in Path(\\"audioset/audio\\").glob(\\"**/*.flac\\")]})\\n", "audioset_dataset = audioset_dataset.cast_column(\\"audio\\", datasets.Audio(sampling_rate=16000))\\n", "for row in tqdm(audioset_dataset):\\n", "    name = row[\'audio\'][\'path\'].split(\'/\')[-1].replace(\\".flac\\", \\".wav\\")\\n", "    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row[\'audio\'][\'array\']*32767).astype(np.int16))\\n", "\\n", "# Free Music Archive dataset (https://github.com/mdeff/fma)\\n", "output_dir = \\"./fma\\"\\n", "if not os.path.exists(output_dir):\\n", "    os.mkdir(output_dir)\\n", "fma_dataset = datasets.load_dataset(\\"rudraml/fma\\", name=\\"small\\", split=\\"train\\", streaming=True)\\n", "fma_dataset = iter(fma_dataset.cast_column(\\"audio\\", datasets.Audio(sampling_rate=16000)))\\n", "\\n", "n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training\\n", "for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips\\n", "    row = next(fma_dataset)\\n", "    name = row[\'audio\'][\'path\'].split(\'/\')[-1].replace(\\".mp3\\", \\".wav\\")\\n", "    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row[\'audio\'][\'array\']*32767).astype(np.int16))\\n", "    i += 1\\n", "    if i == n_hours*3600//30:\\n", "        break\\n"]}, {"cell_type": "code", "execution_count": null, "id": "d01ec467", "metadata": {"id": "d01ec467"}, "outputs": [], "source": ["# Download pre-computed openWakeWord features for training and validation\\n", "\\n", "# training set (~2,000 hours from the ACAV100M Dataset)\\n", "# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information\\n", "!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy\\n", "\\n", "# validation set for false positive rate estimation (~11 hours)\\n", "!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy"]}, {"cell_type": "markdown", "id": "cfe82647", "metadata": {"id": "cfe82647"}, "source": ["# Define Training Configuration"]}, {"cell_type": "markdown", "id": "b2e71329", "metadata": {"id": "b2e71329"}, "source": ["For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.\\n", "\\n", "It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we\'ll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:\\n", "\\n", "- We\'ll train a detection model for the phrase \\"hey sebastian\\"\\n", "- We\'ll only generate 5,000 positive and negative examples (to save on time for this example)\\n", "- We\'ll only generate 1,000 validation positive and negative examples for early stopping (again to save time)\\n", "- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)\\n", "- We\'ll reduce the target metrics to account for the small dataset size and limited training.\\n", "\\n", "On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.\\n"]}, {"cell_type": "code", "execution_count": null, "id": "fb0b6e4f", "metadata": {"ExecuteTime": {"end_time": "2023-09-04T18:11:33.893397Z", "start_time": "2023-09-04T18:11:33.878938Z"}, "id": "fb0b6e4f"}, "outputs": [], "source": ["# Load default YAML config file for training\\n", "config = yaml.load(open(\\"openwakeword/examples/custom_model.yml\\", \'r\').read(), yaml.Loader)\\n", "config"]}, {"cell_type": "code", "execution_count": null, "id": "482cf2d0", "metadata": {"ExecuteTime": {"end_time": "2023-09-04T15:07:00.859210Z", "start_time": "2023-09-04T15:07:00.841472Z"}, "id": "482cf2d0"}, "outputs": [], "source": ["config[\\"target_phrase\\"] = [\\"hey vega\\"]\\n", "config[\\"model_name\\"] = \\"hey_vega_experimental\\"\\n", "config[\\"custom_negative_phrases\\"] = [\\"hey vijay\\", \\"vegan\\", \\"las vegas\\", \\"hey google\\"]\\n", "config[\\"n_samples\\"] = 20000\\n", "config[\\"n_samples_val\\"] = 2000\\n", "config[\\"steps\\"] = 50000\\n", "config[\\"target_accuracy\\"] = 0.9\\n", "config[\\"target_recall\\"] = 0.9\\n", "config[\\"background_paths\\"] = [\\"./audioset_16k\\", \\"./fma\\"]\\n", "config[\\"false_positive_validation_data_path\\"] = \\"validation_set_features.npy\\"\\n", "config[\\"feature_data_files\\"] = {\\"ACAV100M_sample\\": \\"openwakeword_features_ACAV100M_2000_hrs_16bit.npy\\"}\\n", "with open(\\"my_model.yaml\\", \\"w\\") as file:\\n", "    yaml.safe_dump(config, file)\\n"]}, {"cell_type": "markdown", "id": "aa6b2ab0", "metadata": {"id": "aa6b2ab0"}, "source": ["# Train the Model"]}, {"cell_type": "markdown", "id": "a51202c0", "metadata": {"id": "a51202c0"}, "source": ["With the data downloaded and training configuration set, we can now start training the model. We\'ll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process."]}, {"cell_type": "code", "execution_count": null, "id": "f01531fa", "metadata": {"ExecuteTime": {"end_time": "2023-09-04T13:50:08.803326Z", "start_time": "2023-09-04T13:50:06.790241Z"}, "id": "f01531fa"}, "outputs": [], "source": ["# Step 1: Generate synthetic clips\\n", "# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU\\n", "# If generation fails, you can simply run this command again as it will continue generating until the\\n", "# number of files meets the targets specified in the config file\\n", "\\n", "!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips"]}, {"cell_type": "code", "execution_count": null, "id": "afeedae4", "metadata": {"ExecuteTime": {"end_time": "2023-09-04T13:56:08.781018Z", "start_time": "2023-09-04T13:55:40.203515Z"}, "id": "afeedae4"}, "outputs": [], "source": ["# Step 2: Augment the generated clips\\n", "\\n", "!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips"]}, {"cell_type": "code", "execution_count": null, "id": "9ad81ea0", "metadata": {"ExecuteTime": {"end_time": "2023-09-04T15:11:14.742260Z", "start_time": "2023-09-04T15:07:03.755159Z"}, "id": "9ad81ea0"}, "outputs": [], "source": ["# Step 3: Train model\\n", "\\n", "!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model"]}, {"cell_type": "code", "execution_count": null, "id": "JSKWWLalnYzR", "metadata": {"id": "JSKWWLalnYzR"}, "outputs": [], "source": ["# Optional TFLite conversion omitted; evaluate the ONNX artifact locally.\\n"]}, {"cell_type": "markdown", "id": "f9OyUW3ltOSs", "metadata": {"id": "f9OyUW3ltOSs"}, "source": ["After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!"]}], "metadata": {"colab": {"provenance": []}, "kernelspec": {"display_name": "Python 3 (ipykernel)", "language": "python", "name": "python3"}, "language_info": {"codemirror_mode": {"name": "ipython", "version": 3}, "file_extension": ".py", "mimetype": "text/x-python", "name": "python", "nbconvert_exporter": "python", "pygments_lexer": "ipython3", "version": "3.9.12"}, "toc": {"base_numbering": 1, "nav_menu": {}, "number_sections": true, "sideBar": true, "skip_h1_title": false, "title_cell": "Table of Contents", "title_sidebar": "Contents", "toc_cell": false, "toc_position": {}, "toc_section_display": true, "toc_window_display": false}}, "nbformat": 4, "nbformat_minor": 5}'
import urllib.request
assert input("Type DOWNLOAD PUBLIC DATA to prepare assets and execute training: ").strip() == "DOWNLOAD PUBLIC DATA"
def download(url,path):
    path=pathlib.Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if not path.exists():urllib.request.urlretrieve(url,path)
    assert path.stat().st_size > 0
base="https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/"
for asset in ["embedding_model.onnx","embedding_model.tflite","melspectrogram.onnx","melspectrogram.tflite"]:
    download(base+asset,WORK/"openwakeword/openwakeword/resources/models"/asset)
download("https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt",WORK/"piper-sample-generator/models/en_US-libritts_r-medium.pt")
training_notebook = json.loads(PREPARED_NOTEBOOK_JSON)
training_notebook["metadata"]["kernelspec"]={"name":"vega-training-310","display_name":"VEGA Python3.10","language":"python"}
training_path=WORK/"hey_vega_training.ipynb"
training_path.write_text(json.dumps(training_notebook),encoding="utf-8")
env=dict(os.environ); env["PATH"]="/content/vega-env/bin:"+env["PATH"]
subprocess.run([PY,"-m","nbconvert","--to","notebook","--execute","--inplace","--ExecutePreprocessor.kernel_name=vega-training-310","--ExecutePreprocessor.timeout=-1",str(training_path)],cwd=WORK,env=env,check=True)
print("Notebook execution finished. Inspect the executed notebook and actual ONNX artifacts; metrics are not deployment acceptance.")


## Download the experimental result
Keep it separate from the active wake model. Run local validation before configuring VEGA.


In [ ]:
import hashlib
from google.colab import files
models=list(WORK.glob("**/hey_vega_experimental.onnx"))
assert len(models)==1, "Expected exactly one actual trained artifact; inspect training outputs"
model=models[0]
metadata={"status":"experimental_unvalidated","phrase":"Hey Vega","pronunciation":"VAY-guh","sha256":hashlib.sha256(model.read_bytes()).hexdigest(),"openwakeword_commit":"368c03716d1e92591906a84949bc477f3a834455","owner_audio_uploaded":False,"runtime_threshold":None,"next":"Evaluate against untouched local validation recordings, choose threshold, then collect separate final test session"}
meta=model.with_suffix(".metadata.json");meta.write_text(json.dumps(metadata,indent=2))
files.download(str(model));files.download(str(meta));files.download(str(WORK/"environment-freeze.txt"))
